# Feedforward Neural Networks and Backpropagation

## Imports

In [26]:
#Basic libraries
import os
import numpy as np
import copy
import random
import matplotlib.pyplot as plt
import seaborn as sns

#Framework
import torch

## Setting the random seed

In [27]:
torch.manual_seed(0) # Setting the seed

## Tensors
Tensors are the PyTorch equivalent to Numpy arrays, with the addition to also have support for GPU acceleration. The name "tensor" is a generalization of concepts. For instance, a vector is a 1-D tensor, and a matrix a 2-D tensor. When working with neural networks, tensors of various shapes and number of dimensions are used.

Most common functions from numpy can be used on tensors as well. Actually, since numpy arrays are so similar to tensors, tensors could be converted to numpy arrays (and back).

**Initialization**
There are many possible options, the simplest one is to call torch.Tensor passing the desired shape as input argument. The function torch.Tensor allocates memory for the desired tensor, but reuses any values that have already been in the memory. To directly assign values to the tensor during initialization, there are many alternatives including:

1.   torch.zeros: Creates a tensor filled with zeros
2.   torch.ones: Creates a tensor filled with ones
3.   torch.rand: Creates a tensor with random values uniformly sampled between 0 and 1
4.   torch.randn: Creates a tensor with random values sampled from a normal distribution with mean 0 and variance 1
5.   torch.arange: Creates a tensor containing the values  N,N+1,N+2,...,M




In [28]:
x = torch.zeros(2, 3, 4) # 3-D tensor, 2 blocks, each block has 3 rows, each row has 4 numbers
print("\nTensor with all zeroes\n")
print(x)

print("\nTensor with all ones\n")
x = torch.ones(2, 3, 4) # 3-D tensor, 2 blocks, each block has 3 rows, each row has 4 numbers
print(x)


Tensor with all zeroes

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

Tensor with all ones

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])


### Tensor to Numpy, and Numpy to Tensor

Tensors can be converted to numpy arrays, and numpy arrays back to tensors. To transform a numpy array into a tensor, we can use the function `torch.from_numpy`:

In [29]:
np_arr = np.array([[1,2,3], [4,5,6]])
tensor = torch.from_numpy(np_arr) # Creating a tensor from a numpy array

print("\nNumpy array\n")
print(np_arr)

print("\nPytorch tensor\n")
print(tensor)


Numpy array

[[1 2 3]
 [4 5 6]]

Pytorch tensor

tensor([[1, 2, 3],
        [4, 5, 6]])


To transform a PyTorch tensor back to a numpy array, we can use the function `.numpy()` on tensors:

In [30]:
tensor = torch.arange(4) # creates a 1-dimensional tensor, with values starting from 0 up to 4 (but not including 4)
np_arr = tensor.numpy() # convert the PyTorch tensor into a NumPy array, tensor and the NumPy array share the same memory

print("\nPytorch tensor\n")
print(tensor)

print("\nNumpy array\n")
print(np_arr)


Pytorch tensor

tensor([0, 1, 2, 3])

Numpy array

[0 1 2 3]


In [31]:
# Create a tensor on the CPU
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
print("CPU tensor", cpu_tensor)

if torch.cuda.is_available():
    # Move the tensor to the GPU
    gpu_tensor = cpu_tensor.to('cuda')
    print("GPU tensor", gpu_tensor)

    # Move the tensor back to the CPU
    cpu_tensor_again = gpu_tensor.to('cpu')
    print("Tensor moved back to CPU:", cpu_tensor_again)

else:
    print("CUDA is not available. The tensor remains on the CPU.")

CPU tensor tensor([1., 2., 3.])
GPU tensor tensor([1., 2., 3.], device='cuda:0')
Tensor moved back to CPU: tensor([1., 2., 3.])


In [32]:
use_cuda = torch.cuda.is_available() # True if PyTorch can access an NVIDIA GPU with CUDA support, False otherwise
print('Using PyTorch version:', torch.__version__, 'CUDA:', use_cuda)
device = torch.device("cuda" if use_cuda else "cpu")
print("Device in Use:",device)

Using PyTorch version: 2.14.1+cu130 CUDA: True
Device in Use: cuda


In [33]:
# Demonstrate different tensor data types
float_tensor = torch.tensor([1.0,2.0,3.0], dtype=torch.float32)
int_tensor = torch.tensor([1, 2, 3], dtype=torch.int32)
double_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)

print("Float Tensor:", float_tensor)
print("Int Tensor:", int_tensor)
print("Double Tensor:", double_tensor)

Float Tensor: tensor([1., 2., 3.])
Int Tensor: tensor([1, 2, 3], dtype=torch.int32)
Double Tensor: tensor([1., 2., 3.], dtype=torch.float64)


###Basic Tensor Operations

 Indexing, Slicing, Reshaping, Transposing, and Permuting

In [34]:
# Create a tensor
tensor = torch.tensor([[1,2,3], [4,5,6], [7,8,9]])

# Indexing: Accessing specific elements
element = tensor[0, 1] # Access the element in the first row and second column
print(f"Indexed element: {element}")

# Slicing: Accessing a sub-tensor
# :2 means rows from the start up to (but not including) row 2 → rows 0 and 1
# 1: means columns from column 1 to the end → columns 1 and 2
sub_tensor = tensor[:2,1:]
print(f"Sub-tensor:\n{sub_tensor}")

# Reshaping: Changing the shape of the tensor
reshaped_tensor = tensor.view(1,9)
print(f"Reshaped tensor:\n{reshaped_tensor}")

# Transposing: Swapping dimensions
transposed_tensor = tensor.t() # Transpose the tensor
print(f"Transposed tensor:\n{transposed_tensor}")

# Permuting: Rearranging dimensions
# permute changes the order of dimensions
permuted_tensor = tensor.permute(1, 0)  # Permute dimensions (swap axes)
print(f"Permuted tensor:\n{permuted_tensor}")

Indexed element: 2
Sub-tensor:
tensor([[2, 3],
        [5, 6]])
Reshaped tensor:
tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9]])
Transposed tensor:
tensor([[1, 4, 7],
        [2, 5, 8],
        [3, 6, 9]])
Permuted tensor:
tensor([[1, 4, 7],
        [2, 5, 8],
        [3, 6, 9]])


Elementwise Operations

In [35]:
tensor_a = torch.tensor([1,2,3])
tensor_b = torch.tensor([4,5,6])

# Element-wise addition
add_result = tensor_a + tensor_b
print(f"Element-wise addition: {add_result}")

# Element-wise multiplication
mult_result = tensor_a * tensor_b
print(f"Element-wise multiplication: {mult_result}")

# Logical operations, logical comparisons produce True/False values
greater_tensor = tensor_a > tensor_b  # Element-wise comparison
print(f"Greater tensor: {greater_tensor}")

Element-wise addition: tensor([5, 7, 9])
Element-wise multiplication: tensor([ 4, 10, 18])
Greater tensor: tensor([False, False, False])


In [36]:
# Create a tensor
tensor = torch.tensor([1.0, 2.0, 3.0])

# Elementwise operations
sqrt_tensor = torch.sqrt(tensor)  # Element-wise square root
print(f"Square root tensor: {sqrt_tensor}")

exp_tensor = torch.exp(tensor)  # Element-wise exponential
print(f"Exponential tensor: {exp_tensor}")

Square root tensor: tensor([1.0000, 1.4142, 1.7321])
Exponential tensor: tensor([ 2.7183,  7.3891, 20.0855])
